# 4. Structured Output

Contrast with notebook 01's `PydanticOutputParser`: instead of hand-writing format
instructions into the prompt and parsing raw text afterward, `with_structured_output()`
handles both steps for you, binding the schema directly into the model call so the
model is constrained to return data matching it.

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

In [ ]:
from pydantic import BaseModel, Field

from models.chat_models.ollama_models import SupportedModel, get_chat_model


class Recipe(BaseModel):
    name: str = Field(description="The name of the dish.")
    ingredients: list[str] = Field(description="List of ingredients with quantities.")
    steps: list[str] = Field(description="Ordered list of preparation steps.")
    estimated_minutes: int = Field(description="Estimated total time to prepare, in minutes.")


llm = get_chat_model(SupportedModel.llama3_2)
structured_llm = llm.with_structured_output(Recipe)

recipe = structured_llm.invoke("Give me a simple recipe for scrambled eggs.")
print(type(recipe))
print(recipe.model_dump())

In [ ]:
print("name:", recipe.name)
print("estimated_minutes:", recipe.estimated_minutes)
print("ingredients:")
for item in recipe.ingredients:
    print(" -", item)
print("steps:")
for i, step in enumerate(recipe.steps, 1):
    print(f" {i}.", step)

## 🧪 Playground

**1. A different dish** — try something with more steps, e.g. `"beef wellington"`.

In [ ]:
# TODO: structured_llm.invoke with a more complex dish


**2. Add a field** to `Recipe`, e.g. `difficulty: str` ("easy"/"medium"/"hard"), and see if the model fills it in sensibly.

In [ ]:
# TODO: define RecipeV2 with an extra field and re-invoke


**3. A nonsense dish** — try `"a recipe for happiness"` and see how the model (and the schema) handle it.

In [ ]:
# TODO: try a non-food 'dish' and see what comes back
